아래 코드 실행해서 PyTorch 2.3.1 버전 설치 후, 노트북 재시작해주세요.
- 문제점: 최신 PyTorch에서 Kaggle GPU P100(또는 T4)을 지원하지 않음
- 해결방법: 아래 코드 실행 후, 노트북 메뉴 -> Run -> Restart & clear cell outputs
- Restart 이후에, 아래 코드는 다시 실행하지 않음

In [ ]:
!pip install --quiet --upgrade --force-reinstall \
    torch==2.3.1 torchvision==0.18.1 \
    --index-url https://download.pytorch.org/whl/cu121

PyTorch 2.3.1 버전 설치 후, 노트북 재시작한 후에 아래 코드 실행해주세요.

In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
# [변경] 정밀한 최적화를 위한 CosineAnnealingLR 도입
from torch.optim.lr_scheduler import CosineAnnealingLR

from torchvision import datasets, transforms
import torchvision.models as models

from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import numpy as np
import pandas as pd

import os
import random
from tqdm import tqdm
from PIL import Image

In [ ]:
seed_everything(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

In [ ]:
# ==================== [ 해상도 384 확장 및 증강 고도화 ] ====================
IMAGE_SIZE = 384  # 👈 256에서 384로 해상도를 키워 숨은 단서를 찾습니다.

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
base_dataset = datasets.ImageFolder(root='/kaggle/input/competitions/image-classificaion-hbnu-ai-2026-spring/dataset/train')

In [ ]:
# ==================== [ K-Fold 분할 및 배치 다운사이징 ] ====================
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=7)

fold_loaders = []

print(f"🔄 전체 데이터를 {n_splits}개의 Fold로 분할하는 중...")
for fold, (train_idx, val_idx) in enumerate(kf.split(base_dataset)):
    train_subset = Subset(base_dataset, train_idx)
    val_subset = Subset(base_dataset, val_idx)

    train_subset.dataset.transform = train_transform
    val_subset.dataset.transform = val_transform

    # ⚠️ 해상도가 384로 커졌으므로, 안전하게 OOM을 방지하기 위해 배치 크기를 16으로 줄입니다.
    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=2)

    fold_loaders.append((train_loader, val_loader))

print(f"✅ 총 {n_splits}개의 Fold 데이터셋 구성 완료! 안전 배치 크기: 16")

In [ ]:
def get_model():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    num_ftrs = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(num_ftrs, 2)
    return model.to(device)

test_model = get_model()
print("ConvNeXt-Tiny모델 선언 및 탑재 완료!")

In [ ]:
import os
import gc

if not os.path.exists('checkpoint'):
    os.makedirs('checkpoint')

# [변경] 과적합 방지를 위해 레이블 스무딩(0.1)을 주입합니다.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
epochs = 5

print("🚀 [고도화 프리미엄 5-Fold 학습] 최고점을 향해 주행을 시작합니다.")

for fold, (train_loader, val_loader) in enumerate(fold_loaders):
    print(f"\n==================== 📂 FOLD {fold + 1} / {n_splits} 학습 시작 ====================")

    model = get_model()
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # [추가] 에폭이 진행됨에 따라 최적의 학습률로 부드럽게 깎아주는 코사인 스케줄러
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_acc = 0.

    for epoch in range(epochs):
        # --- 훈련 단계 ---
        model.train()
        for inputs, label in tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1} Train"):
            inputs, label = inputs.to(device), label.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), label.long())
            loss.backward()
            optimizer.step()

        # 에폭이 끝날 때마다 학습률 갱신
        scheduler.step()

        # --- 검증 단계 ---
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, label in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                val_preds += predicted.cpu().numpy().tolist()
                val_labels += label.numpy().tolist()

        val_acc = accuracy_score(val_labels, val_preds)
        print(f"📈 [Fold {fold+1}] Epoch {epoch+1} 검증 정확도: {val_acc:.4f} (LR: {scheduler.get_last_lr()[0]:.6f})")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'checkpoint/best_model_fold{fold+1}.pth')
            print(f"⭐ [Fold {fold+1}] 최고 성능 갱신 및 파일 저장 완료.")

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ==================== [ 테스트 단계: 3-Fold 앙상블 + 안전 2중 TTA ] ====================

class SuperEnsembleDataset(Dataset):
    def __init__(self, df, tf_orig, tf_h):
        self.df = df
        self.tf_orig = tf_orig
        self.tf_h = tf_h

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.iloc[idx, 0]
        image_path = f'/kaggle/input/competitions/image-classificaion-hbnu-ai-2026-spring/dataset/test/{image_name}'
        image = Image.open(image_path)
        return self.tf_orig(image), self.tf_h(image)

# 1. 🛠️ [중요] 중간에 멈췄으므로 안전하게 완공된 1, 2, 3번 폴드 모델만 탑재합니다!
models_pool = []
for fold in [0, 1, 2]: # 👈 range(n_splits) 대신 [0, 1, 2]로 직접 지정! (1, 2, 3번 폴드만 로드)
    m = models.convnext_tiny(weights=None)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    m.load_state_dict(torch.load(f'checkpoint/best_model_fold{fold+1}.pth'))
    m = m.to(device).eval()
    models_pool.append(m)

# 2. 해상도 사이즈에 맞춘 테스트 데이터 로더 설정 (val_transform 활용)
test_df = pd.read_csv('/kaggle/input/competitions/image-classificaion-hbnu-ai-2026-spring/submission_sample.csv')
test_dataset = SuperEnsembleDataset(test_df, val_transform, transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0), # 무조건 좌우 반전
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
]))
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

final_predictions = []

# len(models_pool)이 자동으로 3이 되므로, 아래에서 (3 * 2.0 = 6)개 의견의 평균을 공평하게 냅니다.
print(f"📝 [종합 검산] 완공된 {len(models_pool)}개의 최강 폴드 모델이 각각 2중 검산(총 {len(models_pool)*2}개 의견)을 시작합니다...")
with torch.no_grad():
    for img_orig, img_h in tqdm(test_dataloader, desc="3-Fold TTA Inference"):
        img_orig, img_h = img_orig.to(device), img_h.to(device)

        batch_probs = torch.zeros((img_orig.size(0), 2)).to(device)

        # 살아남은 3개의 모델이 각각 원본/반전을 검산
        for model in models_pool:
            batch_probs += torch.softmax(model(img_orig), dim=1)
            batch_probs += torch.softmax(model(img_h), dim=1)

        # 총 6으로 나누어 정밀한 평균 확률 계산
        mean_probs = batch_probs / (len(models_pool) * 2.0)

        _, predicted = torch.max(mean_probs, 1)
        final_predictions += predicted.detach().cpu().numpy().tolist()

# 3. 정답 저장 및 내보내기
test_df['label'] = final_predictions
test_df.to_csv('submission.csv', index=False)
print("✅ Fold 1, 2, 3의 정수만 모은 'submission.csv' 파일이 긴급 생성되었습니다!")